# Phase 3b: Batch Synthetic Data Generation — Colab 40GB

**Goal**: Generate Tamil instruction data faster on Colab using full-precision Llama 3.1 8B Instruct.

**Hardware**: Colab A100 40GB  
**Speed**: ~10K pairs/hour (vs ~500-1000/hour on local 8GB)  

**Use this when**: You need the full 50K pairs quickly, or when local overnight generation isn't enough.

In [ ]:
import subprocess, sys
for pkg in ["unsloth", "transformers", "datasets", "tqdm", "accelerate"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Ready.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
GENERATOR_MODEL  = "meta-llama/Llama-3.1-8B-Instruct"  # full precision on 40GB
BATCH_SIZE       = 8          # number of prompts per forward pass
MAX_NEW_TOKENS   = 300
TEMPERATURE      = 0.7

# Source data (load from HF — Wikipedia articles)
HF_SOURCE        = "wickkiey/tamil-wikipedia-markdown"
N_ARTICLES       = 60000     # sample from source to generate ~50K pairs

OUTPUT_JSONL     = "/content/drive/MyDrive/Tamil-LLM/synthetic_instructions_colab.jsonl"
HF_OUTPUT_REPO   = "wickkiey/tamil-synthetic-instructions"
HF_TOKEN         = None

MIN_TAMIL_RATIO  = 0.40
MIN_RESP_LEN     = 20

print(f"Generator : {GENERATOR_MODEL}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.makedirs("/content/drive/MyDrive/Tamil-LLM", exist_ok=True)
    print("Drive mounted.")
except ImportError:
    print("Not on Colab — output will be saved locally.")
    OUTPUT_JSONL = "./synthetic_instructions_colab.jsonl"

In [ ]:
# ── Load generator model (full precision on 40GB) ──────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL, token=HF_TOKEN)
tokenizer.padding_side = "left"  # for batch generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype  = torch.bfloat16,
    device_map   = "auto",
    token        = HF_TOKEN,
)
model.eval()

print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ── Load source articles ───────────────────────────────────────────────────────
from datasets import load_dataset
import random

source_ds = load_dataset(HF_SOURCE, split="train")
source_ds = source_ds.shuffle(seed=42).select(range(min(N_ARTICLES, len(source_ds))))
articles  = [{"text": x["text"][:600], "title": x.get("title", "")} for x in source_ds]

print(f"Loaded {len(articles):,} articles for generation.")

In [ ]:
# ── Prompt builder ─────────────────────────────────────────────────────────────
def build_qa_prompt(article_text: str) -> str:
    return (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "நீங்கள் ஒரு தமிழ் மொழி வல்லுநர். கொடுக்கப்பட்ட உரையிலிருந்து ஒரு கேள்வி மற்றும் "
        "தெளிவான பதில் தமிழில் எழுதுங்கள். வடிவம்: கேள்வி: <question>\nபதில்: <answer>"
        "<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
        f"உரை:\n{article_text}\n\nகேள்வி மற்றும் பதில் எழுதுங்கள்:"
        "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )

def build_summary_prompt(article_text: str) -> str:
    return (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "நீங்கள் ஒரு தமிழ் மொழி வல்லுநர். கொடுக்கப்பட்ட உரையை 2-3 வாக்கியங்களில் சுருக்கி எழுதுங்கள்."
        "<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
        f"சுருக்கி எழுதுங்கள்:\n{article_text}"
        "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )

print("Prompt builders ready.")

In [ ]:
# ── Batch generation function ──────────────────────────────────────────────────
import json
from tqdm import tqdm

TAMIL_START, TAMIL_END = 0x0B80, 0x0BFF

def tamil_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if TAMIL_START <= ord(c) <= TAMIL_END) / len(text)

@torch.inference_mode()
def generate_batch(prompts: list[str]) -> list[str]:
    inputs = tokenizer(
        prompts,
        return_tensors   = "pt",
        padding          = True,
        truncation       = True,
        max_length       = 1024,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens   = MAX_NEW_TOKENS,
        temperature      = TEMPERATURE,
        do_sample        = True,
        pad_token_id     = tokenizer.eos_token_id,
    )
    # Decode only the generated part (not the prompt)
    return [
        tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        for out in outputs
    ]

print("Batch generator ready.")

In [ ]:
# ── Run batch generation ───────────────────────────────────────────────────────
results = []

# Split articles: 60% Q&A, 40% summarization
qa_articles      = articles[:int(len(articles) * 0.6)]
summary_articles = articles[int(len(articles) * 0.6):]

def run_generation(article_list, prompt_fn, pair_type, label):
    pairs = []
    batches = [article_list[i:i+BATCH_SIZE] for i in range(0, len(article_list), BATCH_SIZE)]
    for batch in tqdm(batches, desc=label):
        prompts   = [prompt_fn(a["text"]) for a in batch]
        responses = generate_batch(prompts)
        for article, response in zip(batch, responses):
            if len(response) < MIN_RESP_LEN:
                continue
            if tamil_ratio(response) < MIN_TAMIL_RATIO:
                continue
            if pair_type == "qa":
                lines    = response.split("\n")
                question = next((l.replace("கேள்வி:", "").strip() for l in lines if "கேள்வி:" in l), "")
                answer   = next((l.replace("பதில்:", "").strip()  for l in lines if "பதில்:"  in l), response)
                if not question:
                    question = f"{article.get('title', 'இது')} பற்றி என்ன தெரியும்?"
                pairs.append({"instruction": question, "input": "", "output": answer, "type": "qa"})
            else:
                pairs.append({
                    "instruction": f"{article.get('title', 'இதை')} பற்றி சுருக்கமாக விளக்குங்கள்.",
                    "input": "", "output": response, "type": "summary"
                })
        # Write incrementally so progress is never lost
        with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
            for p in pairs[-len(batch):]:
                f.write(json.dumps(p, ensure_ascii=False) + "\n")
    return pairs

qa_pairs      = run_generation(qa_articles, build_qa_prompt, "qa", "Q&A")
summary_pairs = run_generation(summary_articles, build_summary_prompt, "summary", "Summary")

all_pairs = qa_pairs + summary_pairs
print(f"\nTotal generated: {len(all_pairs):,}")
print(f"  Q&A     : {len(qa_pairs):,}")
print(f"  Summary : {len(summary_pairs):,}")
print(f"Saved to  : {OUTPUT_JSONL}")

In [ ]:
# ── Push combined dataset to HF ────────────────────────────────────────────────
# Merge with any existing local generation (from notebook 01)
from datasets import Dataset, load_dataset, concatenate_datasets

new_ds = Dataset.from_list(all_pairs)

try:
    existing_ds = load_dataset(HF_OUTPUT_REPO, split="train", token=HF_TOKEN)
    combined_ds = concatenate_datasets([existing_ds, new_ds])
    print(f"Merged with existing {len(existing_ds):,} pairs → total {len(combined_ds):,}")
except Exception:
    combined_ds = new_ds
    print(f"New dataset: {len(combined_ds):,} pairs")

combined_ds = combined_ds.shuffle(seed=42)
combined_ds.push_to_hub(
    HF_OUTPUT_REPO,
    token          = HF_TOKEN,
    commit_message = f"Colab batch generation — added {len(all_pairs):,} pairs"
)
print(f"Pushed to: https://huggingface.co/datasets/{HF_OUTPUT_REPO}")